# LLM-as-a-judge: accuracy and agreement

Compares the level given by an LLM judge with the instructed level: accuracy, macro-F1 and quadratic-weighted Cohen's $\kappa$ with its 95% bootstrap CI (the same functions as for the graph and embedding methods, `lib/evaluation.py`).

Input: the files `..._ratedby_<judge>.csv` (not the `_long` ones), with one row per transcript and the columns
`uid, instruction_id, generated_id, ability|benevolence, instruction_with_intention, instruction, LLMrating_s0, LLMrating_s1, LLMrating_s2, LLMrating, shuffle_agreement, explanation, generator, rater, gender`.

* `ability` / `benevolence`: instructed level (reference);
* `LLMrating`: final level given by the judge (main results);
* `LLMrating_s0`, `_s1`, …: level given in each shuffle of the list, used for the variability across presentation orders.

The task (`nav` or `QA`) and the mode (e.g. `listwise`) are read from the file name, e.g. `..._full_ability_gender_man_mistral_ nav_listwise_ratedby_qwen.csv`.

In [ ]:
import os
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

sys.path.append(os.path.realpath("../lib"))
from evaluation import LEVELS, kappa_with_ci, plot_confusion, quadratic_kappa

## Parameters

In [ ]:
DATA_DIR = r"C:\Users\lucie\G-n-rateur-d-intention\Générateur d'intention\data"
PATTERN = "*_ratedby_*.csv"
OUTPUT_DIR = "results"
DIMENSIONS = ["ability", "benevolence"]
UNPARSABLE = "wrong"            # judge answers that are not a level: "wrong" (count as errors) or "drop"

os.makedirs(OUTPUT_DIR, exist_ok=True)

## Files

In [ ]:
def file_info(path):
    # Task, mode and judge from the end of the file name ("..._ nav_listwise_ratedby_qwen").
    stem = Path(path).stem.replace(" ", "")
    task = "QA" if re.search(r"_QA_", stem, re.IGNORECASE) else "nav"
    mode = re.search(r"_(?:nav|QA)_([A-Za-z]+)_ratedby_", stem, re.IGNORECASE)
    judge = stem.split("_ratedby_", 1)[1]
    return {"task": task, "mode": mode.group(1) if mode else "", "judge": judge}


files = [f for f in sorted(Path(DATA_DIR).glob(PATTERN))
         if not f.stem.endswith("_long") and "_ratedby_" in f.stem]
pd.DataFrame([{"file": f.name, **file_info(f)} for f in files])

## Reading the judgments

Levels are normalized: `Low` / `Medium` / `High` in any case or inside a sentence, or `1` / `2` / `3`. The table shows, for each file, the share of judge answers that could not be read as a level.

In [ ]:
LEVEL_WORDS = re.compile(r"\b(low|medium|high)\b", re.IGNORECASE)


def to_level(value):
    # Low / Medium / High from a judge answer, or None if it is not a single level.
    if pd.isna(value):
        return None
    text = str(value).strip()
    if text in {"1", "2", "3", "1.0", "2.0", "3.0"}:
        return LEVELS[int(float(text)) - 1]
    words = {w.capitalize() for w in LEVEL_WORDS.findall(text)}
    return words.pop() if len(words) == 1 else None


frames, report = [], []
for path in files:
    data = pd.read_csv(path)
    info = file_info(path)
    shuffle_columns = sorted(c for c in data.columns if re.fullmatch(r"LLMrating_s\d+", c))
    for dimension in DIMENSIONS:
        if dimension not in data.columns:
            continue
        frame = pd.DataFrame({
            "uid": data["uid"],
            "true": data[dimension].map(to_level),
            "raw": data["LLMrating"],
            "pred": data["LLMrating"].map(to_level),
            **{c.replace("LLMrating_", "pred_"): data[c].map(to_level) for c in shuffle_columns},
            "shuffle_agreement": data.get("shuffle_agreement"),
            "dimension": dimension,
            "explanation": data.get("explanation", ""),
            "generator": data.get("generator", ""),
            "gender": data.get("gender", ""),
            "judge": data["rater"] if "rater" in data else info["judge"],
            "task": info["task"],
            "mode": info["mode"],
        })
        frame = frame[frame["true"].notna()]
        report.append({"file": path.name, "dimension": dimension, "transcripts": len(frame),
                       "shuffles": len(shuffle_columns), "unparsable": frame["pred"].isna().mean()})
        frames.append(frame)

judgments = pd.concat(frames, ignore_index=True)
if UNPARSABLE == "drop":
    judgments = judgments[judgments["pred"].notna()]
else:
    judgments["pred"] = judgments["pred"].fillna("unparsable")
pd.DataFrame(report).round(3)

Answers that could not be read as a level (if any, adjust `to_level`):

In [ ]:
judgments.loc[judgments["pred"] == "unparsable", "raw"].astype(str).value_counts().head(20)

## Accuracy and agreement

* `accuracy`, `macro_f1`: on all transcripts (unparsable answers count as errors with `UNPARSABLE = "wrong"`);
* `kappa [95% CI]`: quadratic-weighted Cohen's $\kappa$ between `LLMrating` and the instructed level, 95% percentile bootstrap CI over transcripts (1000 resamples), on the answers that are a level;
* `kappa per shuffle`: mean ± SD of $\kappa$ computed with each shuffle alone (`LLMrating_s0`, …): sensitivity to the order in which the transcripts were presented;
* `shuffle_agreement`: mean of the column of the same name.

In [ ]:
INDEX = {level: k for k, level in enumerate(LEVELS)}


def kappa_of(true, pred):
    keep = pred.isin(LEVELS)
    return quadratic_kappa(true[keep].map(INDEX).to_numpy(), pred[keep].map(INDEX).to_numpy())


def summary(data, by):
    rows = []
    for keys, part in data.groupby(by):
        keys = keys if isinstance(keys, tuple) else (keys,)
        readable = part[part["pred"].isin(LEVELS)]
        per_shuffle = [kappa_of(part["true"], part[c]) for c in part.columns if re.fullmatch(r"pred_s\d+", c)
                       and part[c].notna().any()]
        kappa = kappa_with_ci(readable)
        rows.append({**dict(zip(by, keys)), "n": len(part),
                     "unparsable": 1 - len(readable) / len(part),
                     "accuracy": (part["pred"] == part["true"]).mean(),
                     "macro_f1": f1_score(part["true"], part["pred"], labels=LEVELS, average="macro"),
                     "kappa": kappa["kappa"],
                     "kappa [95% CI]": f"{kappa['kappa']:.2f} [{kappa['kappa_ci_low']:.2f}, {kappa['kappa_ci_high']:.2f}]",
                     "kappa per shuffle": (f"{np.mean(per_shuffle):.2f} ± {np.std(per_shuffle, ddof=1):.2f}"
                                           if len(per_shuffle) > 1 else ""),
                     "shuffle_agreement": pd.to_numeric(part["shuffle_agreement"], errors="coerce").mean()})
    return pd.DataFrame(rows)


overall = summary(judgments, ["judge", "mode", "dimension"])
overall.to_csv(os.path.join(OUTPUT_DIR, "llm_judge_results.csv"), index=False)
overall.round(3)

By generator, gender and task:

In [ ]:
detailed = summary(judgments, ["judge", "mode", "dimension", "generator", "gender", "task"])
detailed.to_csv(os.path.join(OUTPUT_DIR, "llm_judge_results_by_condition.csv"), index=False)
detailed.round(3)

## Confusion matrices

One per judge, mode and dimension (rows: instructed level; unparsable answers left out).

In [ ]:
for (judge, mode), part in judgments.groupby(["judge", "mode"]):
    predictions = {(dimension, f"{judge} {mode}".strip()): p[p["pred"].isin(LEVELS)]
                   for dimension, p in part.groupby("dimension")}
    plot_confusion(predictions, "instructed", os.path.join(OUTPUT_DIR, f"llm_judge_confusion_{judge}_{mode}.png"))
    plt.show()